<a href="https://colab.research.google.com/github/mahmoudabdelaziz120/FLYRANK_ML-1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:
import duckdb
from google.colab import userdata

# 1. جلب التوكن من Colab Secrets
hf_token = userdata.get('HF_TOKEN').strip()

# 2. إنشاء اتصال DuckDB وتفعيل خاصية الاتصال بـ Hugging Face
con = duckdb.connect(database=':memory:')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}');")

# 3. قراءة شريحة شهر مارس 2026 لجدول الأداء اليومي مباشرة
# نحدد شهر 2026-03 كما هو مطلوب في التكليف (mid-panel month)
dataset_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# تسجيل البيانات كجدول افتراضي داخل DuckDB
con.execute(f"""
    CREATE OR REPLACE VIEW gsc_slice AS
    SELECT * FROM read_parquet('{dataset_path}')
""")

# 4. التحقق من نجاح التحميل واستعراض الأعمدة وعدد الصفوف
row_count = con.execute("SELECT COUNT(*) FROM gsc_slice").fetchone()[0]
cols = [col[0] for col in con.execute("DESCRIBE gsc_slice").fetchall()]

print(f"Loaded successfully! Total rows in March 2026: {row_count:,}")
print("Columns:", cols)

Loaded successfully! Total rows in March 2026: 9,841,378
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [3]:
# فحص هيكل الجدول وتصنيف الأعمدة برمجياً
schema_info = con.execute("DESCRIBE gsc_slice").df()

# عرض أسماء الأعمدة وأنواع بياناتها للتأكد من مطابقتها للتصنيف
display_cols = schema_info[['column_name', 'column_type']]
print("--- Schema Overview ---")
print(display_cols.to_string(index=False))

# عينة من السجلات للتأكد من شكل البيانات الخام
sample_df = con.execute("""
    SELECT
        *
    FROM gsc_slice
    LIMIT 5
""").df()

print("\n--- Sample Data Preview (First 5 rows) ---")
sample_df.head()

--- Schema Overview ---
             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# =====================================================================
# 1. إثبات الـ Grain (عدم وجود تكرار للمفتاح المركب)
# =====================================================================
q_grain = """
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS duplicates_count
FROM gsc_slice
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
"""
dup_check = con.execute(q_grain).fetchall()
print(f"Fact 1 (Grain Proof): Found {len(dup_check)} duplicate rows.")
assert len(dup_check) == 0, "Grain claim failed: duplicate rows detected!"

# =====================================================================
# 2. إثبات النطاق الزمني والعدد الإجمالي للصفوف (Counts & Date Span)
# =====================================================================
q_boundaries = """
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    COUNT(DISTINCT content_hash_id) AS distinct_contents
FROM gsc_slice
"""
df_bounds = con.execute(q_boundaries).df()
print("\nFact 2 (Row Count & Temporal Boundaries):")
print(df_bounds.to_string(index=False))

# =====================================================================
# 3. التحقق من التوافر الصريح باستخدام IS TRUE
# =====================================================================
q_availability = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) AS available_rows,
    ROUND(COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) AS survival_pct
FROM gsc_slice
"""
df_avail = con.execute(q_availability).df()
print("\nFact 3 (Availability Filter with IS TRUE):")
print(df_avail.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fact 1 (Grain Proof): Found 0 duplicate rows.

Fact 2 (Row Count & Temporal Boundaries):
 total_rows   min_date   max_date  distinct_clients  distinct_contents
    9841378 2026-03-01 2026-03-31                55             331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Fact 3 (Availability Filter with IS TRUE):
 total_rows  available_rows  survival_pct
    9841378         3611061         36.69


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Inherent Slice Limitations & Blindspots

1. **Unbalanced Panel History & Survivorship Bias:**
   * Not all clients onboarded simultaneously; each client has distinct start dates (`gsc_data_start` vs `ga4_data_start`).
   * A cross-sectional slice (e.g., `month=2026-03`) contains uneven rolling depths across clients. New clients lack a genuine 7-day or 30-day historical baselineHere is the breakdown of what this dataset cannot tell you, why those blind spots exist, and where analytical assumptions will break.

---

### 1. Unbalanced History
When historical data depth varies across entities, segments, or timeframes, the data fundamentally conceals baseline behavioral shifts:

* **True Causality vs. Survivorship Bias:** You cannot distinguish whether an entity’s high performance is structural or simply the result of surviving long enough to be recorded. Short-history entities may appear volatile or underperforming simply because they lack the smoothing effects of long-run aggregation.
* **Pre-Tracking Baselines:** You cannot establish whether an early trend is an anomaly or a return to mean. If historical tracking begins mid-lifecycle, you cannot infer initial growth velocity, foundational churn drivers, or baseline conversion efficiency.
* **Stationarity and Seasonality:** You cannot reliably test for cyclicality or stationarity. Attempting time-series decomposition (e.g., Fourier transforms, seasonal ARIMA) on unbalanced panels yields spurious coefficients, as short-history series mistake localized macro shocks for intrinsic seasonality.

---

### 2. GSC-Only (Google Search Console) Early Rows
Relying strictly on early-stage Google Search Console data introduces systemic voids in intent and behavioral attribution:

* **Zero Post-Click Realities:** GSC records impressions, clicks, CTR, and average position; it terminates at the click. It cannot tell you bounce rate, session duration, scroll depth, conversion rate, or revenue. A surge in clicks could represent qualified demand or catastrophic misalignment with user intent.
* **Cannibalization vs. Aggregation:** Early rows cannot confirm single-page ranking dominance versus multi-page internal conflict. GSC aggregates position by property and query; without integrated landing page telemetry, you cannot observe query cannibalization or bounce loops across sister URLs.
* **Missing Search Volume Realities (The Long-Tail Mask):** Due to Google’s privacy thresholds and anonymization filters, GSC omits very low-volume queries. A drop in reported query diversity in early rows does not prove demand consolidated; it often means search activity fragmented into anonymized sub-threshold variants.
* **SERP Feature Context:** An "Average Position: 2.1" in GSC tells you nothing about whether the snippet appeared beneath an AI Overview, Local Pack, Knowledge Panel, or four Sponsored Ads. It cannot measure true pixel real estate or organic visibility decay.

---

### 3. Window Overlaps
When analytical windows, attribution lookbacks, or feature-engineering rolling periods overlap, structural noise masquerades as signal:

* **Independence of Events:** Overlapping windows violate the assumption of independent and identically distributed ($i.i.d.$) samples. The data cannot tell you if multiple "events" are distinct reactions or the continuous echo of a single upstream event measured twice.
* **Double-Counted Attribution:** In attribution modeling or rolling aggregations (e.g., a 14-day rolling click-to-conversion window evaluated weekly), a single conversion event can appear inside multiple reporting frames. This inflates apparent pipeline volume and distorts cost-per-acquisition (CPA) calculations.
* **Lead-Lag Causality:** You cannot isolate true lead times. Overlaps blur the exact boundary where an input (e.g., an optimization, a technical release, or a bid change) directly influenced an output, creating artificial autocorrelation and masking the system’s true settling time.

---

### The Blind Spots at Their Intersection

| Data Defect | What the Data Shows | What It Can Never Confirm | Failure Mode If Ignored |
| :--- | :--- | :--- | :--- |
| **Unbalanced History** | Trend lines diverging over time | Equal operating conditions across entities | Misallocating resources to mature entities over viable new ones |
| **GSC-Only Early Rows** | Search demand and top-line organic traffic | Commercial value, intent alignment, or user satisfaction | Optimizing for high-impression, zero-monetization queries |
| **Window Overlaps** | Smooth curves, persistent correlations | Independent validation of outcomes | Overfitting models to self-correlated statistical artifacts |

What specific downstream task (e.g., feature engineering, causal inference, or reporting attribution) are you structuring this data pipeline to support?

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.